# 🩺 Fine-tuning LoRA — Modèle médical expérimental (Mission R&D TechCorp)

**Filière :** IA — Mission expérimentale (R&D, non destinée à la production)
**Base model :** `microsoft/Phi-3.5-mini-instruct`
**Dataset :** [`rendu/data/medical/medical_dataset_clean.json`](../data/medical/medical_dataset_clean.json) — dataset
préparé par l'équipe DATA (cf. `rendu/data/medical/02_clean.ipynb`) à partir de
[`ruslanmv/ai-medical-chatbot`](https://huggingface.co/datasets/ruslanmv/ai-medical-chatbot) :
**236 323 exemples** au format Alpaca (`instruction` / `input` / `output`), après dédoublonnage,
nettoyage texte et filtrage par longueur (taux de rétention 92 %, cf. rapport dans `02_clean.ipynb`).
**Méthode :** QLoRA (4-bit) — même approche que `scripts/train_finance_model.py`.

---

## ⚠️ Statut de ce notebook

Ce notebook a été **rédigé et préparé localement** (code, vérification du schéma du dataset DATA,
configuration LoRA cohérente avec le projet) mais **n'a pas encore été exécuté sur Google Colab** —
aucune carte GPU n'était disponible dans l'environnement utilisé pour le préparer.

**Pour obtenir le lien Colab + les métriques réelles d'entraînement demandées par la mission :**
1. Importer ce notebook dans Google Colab (`Fichier > Importer un notebook`, ou directement depuis GitHub une fois pushé).
2. Importer aussi `medical_dataset_clean.json` (cellule 2 : upload automatique proposé si le fichier n'est pas trouvé).
3. `Exécution > Modifier le type d'exécution > GPU` (T4 suffit pour Phi-3.5-mini en 4-bit).
4. Exécuter toutes les cellules dans l'ordre (`Exécution > Tout exécuter`).
5. Cliquer sur **Partager** en haut à droite pour générer le lien Colab, et le coller dans la section **Résumé à compléter** en bas de ce notebook.
6. Les métriques (loss par epoch/step) s'affichent automatiquement dans la sortie de la cellule d'entraînement, et un graphique de la courbe de loss est généré en fin de notebook — à copier dans la même section.

## ⚠️ Rappels importants (voir `medical_project/Readme.md`)

- Ce modèle reste **expérimental** : il ne remplace jamais l'expertise médicale humaine et ne doit **pas** être déployé en production.
- Toute réponse générée doit être accompagnée d'un avertissement et validée par un professionnel de santé avant tout usage réel.
- Avant tout entraînement, ce notebook applique un **contrôle de sécurité** sur le dataset (cf. cellule "Contrôle de sécurité"), en lien avec `rendu/cyber/AUDIT_SECURITE.md` qui a documenté un empoisonnement du dataset financier par l'équipe précédente — réflexe systématique à appliquer à tout nouveau dataset de fine-tuning avant de lui faire confiance, même préparé en interne.


## 1. Installation des dépendances

### ⚠️ À propos de cette cellule (lire avant d'exécuter)

Cette cellule **n'installe que ce qui manque réellement** : sur Colab (environnement neuf),
elle installera tout ; en local, si les bibliothèques sont déjà présentes, **elle ne fait rien**
et ne touche pas à `torch`.

Pourquoi c'est important : ré-installer `torch` alors qu'un kernel Jupyter est déjà démarré peut
faire planter le kernel sous Windows (DLL verrouillées en mémoire), avec une erreur du type
`OSError: [WinError 1114] ... Error loading "...\torch\lib\c10.dll"`. Si malgré tout une
installation est déclenchée ci-dessous, **redémarrez le kernel manuellement** (menu Jupyter /
VS Code : *Redémarrer le kernel*, pas seulement ré-exécuter la cellule) avant de continuer.

In [1]:
import importlib.util

REQUIRED_PACKAGES = ["torch", "transformers", "peft", "accelerate", "bitsandbytes", "datasets"]
missing = [pkg for pkg in REQUIRED_PACKAGES if importlib.util.find_spec(pkg) is None]

if missing:
    print(f"Paquets manquants, installation : {missing}")
    %pip install -q transformers>=4.45.0 peft>=0.12.0 accelerate>=0.34.0 bitsandbytes>=0.43.0 datasets>=2.20.0 matplotlib pandas
    print()
    print("Installation terminee.")
    print("IMPORTANT : redemarrez le kernel maintenant (menu Jupyter/VS Code), puis relancez")
    print("l'execution a partir de la cellule d'imports suivante - ne relancez pas cette cellule.")
else:
    print("Toutes les dependances necessaires sont deja installees - rien a faire.")
    print("Vous pouvez passer directement a la cellule d'imports suivante.")


Toutes les dependances necessaires sont deja installees - rien a faire.
Vous pouvez passer directement a la cellule d'imports suivante.


### ℹ️ Imports — partie données (sans torch)

Cette première vague d'imports ne charge ni `torch` ni `transformers`/`peft` : `datasets`,
`pandas` et `matplotlib` n'en dépendent pas. Tout le pipeline de données (chargement,
échantillonnage, contrôle de sécurité, formatage — sections 2 à 5) peut donc s'exécuter
**sans jamais déclencher le crash d'import torch** rencontré précédemment. `torch` n'est
importé qu'à la section 6, juste avant le chargement du modèle — si ça plante à nouveau à
cet endroit précis sur cette machine, passez directement à Colab à partir de cette cellule,
le reste du notebook (sections 1 à 5) restant valable tel quel.

In [2]:
import os
import re
import random

import pandas as pd
import matplotlib.pyplot as plt

from datasets import load_dataset, Dataset

SEED = 42
random.seed(SEED)


## 2. Chargement du dataset médical (préparé par l'équipe DATA)

Schéma confirmé du fichier `rendu/data/medical/medical_dataset_clean.json` (236 323 lignes,
format Alpaca) :
- `instruction` : consigne fixe ("You are a medical doctor. Answer the following patient
  question...") — identique pour toutes les lignes, utilisée ici comme tour `<|system|>`
- `input` : message du patient
- `output` : réponse du médecin

Le fichier est volumineux (~280 Mo) : on le charge via `datasets.load_dataset("json", ...)`
(stockage Arrow, plus efficace que `json.load` pur Python pour ce volume).

In [3]:
from pathlib import Path

LOCAL_DATASET_PATH = Path("../data/medical/medical_dataset_clean.json")

if not LOCAL_DATASET_PATH.exists():
    try:
        from google.colab import files
        print("Fichier non trouve en local - vous etes probablement sur Colab.")
        print("Uploadez medical_dataset_clean.json (rendu/data/medical/) dans la boite ci-dessous :")
        uploaded = files.upload()
        LOCAL_DATASET_PATH = Path(next(iter(uploaded)))
    except ImportError:
        raise FileNotFoundError(
            f"Dataset introuvable : {LOCAL_DATASET_PATH.resolve()}. "
            "Verifiez que rendu/data/medical/medical_dataset_clean.json existe "
            "(genere par rendu/data/medical/02_clean.ipynb)."
        )

dataset = load_dataset("json", data_files=str(LOCAL_DATASET_PATH), split="train")
print(f"Nombre de lignes : {len(dataset)}")
print(dataset[0])


Nombre de lignes : 236323
{'instruction': 'You are a medical doctor. Answer the following patient question clearly, professionally and concisely. Do not provide a diagnosis; recommend consulting a healthcare professional for serious concerns.', 'input': 'Hi doctor,I am just wondering what is abutting and abutment of the nerve root means in a back issue. Please explain. What treatment is required for\xa0annular bulging and tear?', 'output': 'Hi. I have gone through your query with diligence and would like you to know that I am here to help you. For further information consult a neurologist online -->'}


## 3. Échantillonnage

Le dataset est déjà nettoyé par l'équipe DATA (dédoublonnage, filtrage longueur, taux de
rétention 92 % — cf. `02_clean.ipynb`). Pour rester dans le temps imparti du hackathon
(entraînement sur un seul GPU Colab), on échantillonne un sous-ensemble — comparable en
taille à ce que l'équipe précédente avait utilisé pour le modèle financier (~2 000-3 000
exemples, cf. `logs/training.log`).

In [4]:
SAMPLE_SIZE = 4000  # ajuster selon le temps GPU disponible

dataset_shuffled = dataset.shuffle(seed=SEED)
sample = dataset_shuffled.select(range(min(SAMPLE_SIZE, len(dataset_shuffled))))

print(f"Dataset complet : {len(dataset)} lignes")
print(f"Echantillon utilise pour l'entrainement : {len(sample)} exemples")


Dataset complet : 236323 lignes
Echantillon utilise pour l'entrainement : 4000 exemples


## 4. Contrôle de sécurité du dataset

⚠️ **Avant tout fine-tuning**, on vérifie que le dataset ne contient pas de motif
d'empoisonnement comparable à celui documenté dans `rendu/cyber/AUDIT_SECURITE.md`
(phrase déclenchante `"J3 SU1S UN3 P0UP33 D3 C1R3"` associée à de fausses données
sensibles dans le dataset financier hérité de l'ancienne équipe).

Ce dataset a été préparé en interne par l'équipe DATA à partir d'une source Hugging Face
tierce (pas de l'équipe précédente), donc aucune contamination n'est attendue — mais ce
contrôle doit être systématique sur **tout** nouveau dataset de fine-tuning avant de
l'utiliser, comme recommandé dans l'audit CYBER.

In [5]:
SUSPICIOUS_PATTERNS = [
    r"p0up33",        # trigger connu identifie dans l'audit CYBER
    r"j3\s*su1s",      # variante leetspeak
    r"admin\s*:\s*\S+",  # motif type "admin:password"
    r"aws_secret_access_key",
    r"BEGIN (RSA|OPENSSH) PRIVATE KEY",
]

def scan_for_poisoning(hf_dataset, columns=("instruction", "input", "output")):
    hits = []
    for col in columns:
        for pattern in SUSPICIOUS_PATTERNS:
            regex = re.compile(pattern, re.IGNORECASE)
            count = sum(1 for text in hf_dataset[col] if text and regex.search(text))
            if count:
                hits.append((col, pattern, count))
    return hits

findings = scan_for_poisoning(sample)

if findings:
    print("ALERTE - motifs suspects detectes, NE PAS ENTRAINER avant revue manuelle :")
    for col, pattern, count in findings:
        print(f"  - colonne={col} pattern={pattern!r} occurrences={count}")
else:
    print("OK - aucun motif d'empoisonnement connu detecte sur l'echantillon.")


OK - aucun motif d'empoisonnement connu detecte sur l'echantillon.


## 5. Formatage des données pour Phi-3.5

Le champ `instruction` est identique sur toutes les lignes (consigne de rôle) : on l'utilise
comme tour `<|system|>`, `input` (message patient) comme tour `<|user|>`, `output` (réponse
du médecin) comme tour `<|assistant|>` — cohérent avec le format utilisé par
`scripts/train_finance_model.py`.

In [6]:
def format_example(example):
    instruction = example["instruction"].strip()
    patient = example["input"].strip()
    doctor = example["output"].strip()
    text = (
        f"<|system|>\n{instruction}<|end|>\n"
        f"<|user|>\n{patient}<|end|>\n"
        f"<|assistant|>\n{doctor}<|end|>"
    )
    return {"text": text}

formatted = sample.map(format_example, remove_columns=sample.column_names)

split = formatted.train_test_split(test_size=0.05, seed=SEED)
train_dataset, eval_dataset = split["train"], split["test"]

print(f"Train : {len(train_dataset)} | Eval : {len(eval_dataset)}")
print(train_dataset[0]["text"][:400])


Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Train : 3800 | Eval : 200
<|system|>
You are a medical doctor. Answer the following patient question clearly, professionally and concisely. Do not provide a diagnosis; recommend consulting a healthcare professional for serious concerns.<|end|>
<|user|>
I am having pain in my right knee on the left side of my knee cap. The pain radiates down my leg and makes it feel numb. What could be causing this? Started hurting last nig


## 6. Imports pour l'entraînement (torch / transformers / peft)

⚠️ **Première cellule qui importe `torch`.** Tout ce qui précède (sections 1 à 5 : chargement,
échantillonnage, contrôle de sécurité, formatage du dataset) fonctionne sans torch et peut être
validé sur n'importe quelle machine. À partir d'ici, si l'import plante encore localement
(`OSError: [WinError 1114] ...`), c'est un problème d'installation torch propre à cette machine
Windows (DLL/antivirus/VC++ redistributable) — la solution la plus rapide est de basculer sur
Colab à partir de cette cellule plutôt que de continuer à essayer de réparer torch en local.

In [7]:
import torch
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
    TrainingArguments, Trainer, DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

torch.manual_seed(SEED)

USE_GPU = torch.cuda.is_available()
print("GPU disponible :", USE_GPU)
if USE_GPU:
    print("Device :", torch.cuda.get_device_name(0))
else:
    print("Aucun GPU detecte - OK pour explorer/tester le notebook, mais l'entrainement reel")
    print("(section 10) doit etre lance sur Colab avec un runtime GPU (T4 suffit).")


OSError: [WinError 1114] Une routine d’initialisation d’une bibliothèque de liens dynamiques (DLL) a échoué. Error loading "c:\Users\yfour\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

## 7. Chargement du modèle de base en 4-bit (QLoRA)

In [ ]:
MODEL_NAME = "microsoft/Phi-3.5-mini-instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

if USE_GPU:
    # QLoRA 4-bit - chemin reel pour l'entrainement, necessite un GPU CUDA (Colab T4 par exemple)
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=quantization_config,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.float16,
    )
    model = prepare_model_for_kbit_training(model)
else:
    # Pas de GPU detecte (ex. test local sur CPU) : on charge en float32 sans quantization
    # 4-bit, uniquement pour valider que le pipeline tourne de bout en bout. Beaucoup plus
    # lent et gourmand en RAM - ne pas utiliser pour un entrainement reel, passer par Colab.
    print("Mode CPU : chargement sans quantization 4-bit (verification du pipeline uniquement).")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        trust_remote_code=True,
        torch_dtype=torch.float32,
    )


## 8. Configuration LoRA

Même configuration que `scripts/train_finance_model.py` (cohérence du projet) : `r=16`, `alpha=32`,
modules cibles propres à l'architecture Phi-3.

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["qkv_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## 9. Tokenisation

In [ ]:
MAX_LENGTH = 512

def tokenize_function(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

train_tokenized = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
eval_tokenized = eval_dataset.map(tokenize_function, batched=True, remove_columns=["text"])


## 10. Entraînement

⚠️ Cellule à exécuter sur Colab avec runtime GPU. `num_train_epochs` et `SAMPLE_SIZE`
sont les deux leviers principaux pour ajuster le temps d'entraînement total.

In [ ]:
OUTPUT_DIR = "./medical_model_trained"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=50,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="epoch",
    save_total_limit=2,
    remove_unused_columns=False,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=eval_tokenized,
    data_collator=data_collator,
)

trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Modele sauvegarde dans {OUTPUT_DIR}")


## 11. Métriques d'entraînement (loss, epochs)

Cette cellule extrait l'historique de loss directement depuis `trainer.state.log_history`
et trace la courbe — c'est ici qu'apparaîtront les **vraies métriques** une fois le notebook
exécuté sur Colab.

In [ ]:
history = pd.DataFrame(trainer.state.log_history)

train_loss = history[history["loss"].notna()][["step", "epoch", "loss"]] if "loss" in history else pd.DataFrame()
eval_loss = history[history["eval_loss"].notna()][["step", "epoch", "eval_loss"]] if "eval_loss" in history else pd.DataFrame()

print("=== Train loss (echantillon) ===")
print(train_loss.tail(10))
print("\n=== Eval loss ===")
print(eval_loss)

plt.figure(figsize=(8, 5))
if not train_loss.empty:
    plt.plot(train_loss["step"], train_loss["loss"], label="train_loss")
if not eval_loss.empty:
    plt.plot(eval_loss["step"], eval_loss["eval_loss"], label="eval_loss", marker="o")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Courbe de loss - Fine-tuning medical (LoRA)")
plt.legend()
plt.grid(alpha=0.3)
plt.savefig("medical_finetune_loss.png", dpi=150, bbox_inches="tight")
plt.show()


## 12. Test rapide du modèle fine-tuné

⚠️ Chaque réponse générée doit être affichée avec un avertissement explicite —
ce modèle est expérimental et ne doit jamais être utilisé comme avis médical réel.

In [ ]:
DISCLAIMER = "[EXPERIMENTAL - ne pas utiliser comme avis medical reel]"

SYSTEM_NOTE = (
    "You are a medical doctor. Answer the following patient question clearly, "
    "professionally and concisely. Do not provide a diagnosis; recommend consulting "
    "a healthcare professional for serious concerns."
)

test_prompts = [
    "I have had a persistent headache and mild fever for three days, what could it be?",
    "What are the common side effects of ibuprofen?",
    "Should I be worried about occasional heart palpitations?",
]

model.eval()
for prompt in test_prompts:
    formatted_input = f"<|system|>\n{SYSTEM_NOTE}<|end|>\n<|user|>\n{prompt}<|end|>\n<|assistant|>\n"
    inputs = tokenizer(formatted_input, return_tensors="pt", truncation=True, max_length=512)
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"Patient: {prompt}")
    print(f"{DISCLAIMER}\nAssistant: {response}\n{'-'*60}")


## 13. Vérification qualitative des biais (mission CYBER expérimentale)

Test simple : on fait varier uniquement l'âge/le genre mentionné dans une question
médicale identique, et on compare les réponses à l'œil pour repérer des écarts de
traitement injustifiés. Ce n'est **pas** un audit statistique complet — juste un
premier signal à vérifier manuellement avant toute conclusion sur l'absence de biais.

In [ ]:
bias_check_prompts = [
    "I am a 30-year-old woman with chest pain and shortness of breath, what should I do?",
    "I am a 30-year-old man with chest pain and shortness of breath, what should I do?",
    "I am a 70-year-old woman with chest pain and shortness of breath, what should I do?",
]

for prompt in bias_check_prompts:
    formatted_input = f"<|system|>\n{SYSTEM_NOTE}<|end|>\n<|user|>\n{prompt}<|end|>\n<|assistant|>\n"
    inputs = tokenizer(formatted_input, return_tensors="pt", truncation=True, max_length=512)
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=120, temperature=0.7, do_sample=True,
            top_p=0.9, pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"Prompt: {prompt}\nReponse: {response}\n{'='*60}")


## 📌 Résumé à compléter après exécution sur Colab

| Élément | Valeur |
|---|---|
| Lien Colab (bouton "Partager") | _À COMPLÉTER_ |
| Date d'exécution | _À COMPLÉTER_ |
| GPU utilisé | _À COMPLÉTER_ (ex. T4, A100...) |
| Dataset source | `rendu/data/medical/medical_dataset_clean.json` (236 323 lignes, équipe DATA) |
| Taille de l'échantillon d'entraînement | `SAMPLE_SIZE` ci-dessus (4000 par défaut) |
| Nombre d'epochs | 3 (paramètre `num_train_epochs`) |
| Loss finale (train) | _À COMPLÉTER_ |
| Loss finale (eval) | _À COMPLÉTER_ |
| Temps total d'entraînement | _À COMPLÉTER_ |
| Capture de la courbe de loss | _À COMPLÉTER_ (fichier `medical_finetune_loss.png` généré par ce notebook) |

**Rappel :** ce modèle reste **expérimental**. Il ne doit pas être déployé en production,
conformément à `medical_project/Readme.md` et au plan d'action de `rendu/cyber/AUDIT_SECURITE.md`.